# Chapter 18 – Worked examples

This notebook runs the examples of [Chapter 18](../index.md) live on IEEE 14. The code lives in the modules
next to it (`ieee14.py`, `bsdf.py`, `modf.py`, `compose.py`, `splits.py`); the cells only call them, so a
change in a module is a change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
import compose
import modf
import numpy as np
import splits
from ieee14 import load_case14

grid = load_case14()
h = grid.ptdf()
print(grid.n_bus, 'buses,', grid.n_branch, 'branches; base-case flows in MW:')
for name, flow in zip(grid.names, grid.flows(), strict=True):
    print(f'{name:7s} {flow:8.2f}')

## Disconnections and multi-outages (MODF)

The joint outage of two branches solves a $2 \times 2$ system instead of two scalar LODF equations. The
result equals the PTDF of the grid without those branches; the determinant of $I - \mathcal H_{OO}$ is the
islanding check.

In [ ]:
out = [grid.names.index('L2-4'), grid.names.index('L4-5')]
m, det = modf.modf(h, grid.frm, grid.to, out)
h_modf = modf.apply_modf(h, grid.frm, grid.to, out)
keep = [k for k in range(grid.n_branch) if k not in out]
print('det(I - H_OO) =', round(det, 6))
print('max |H_modf - H_recomputed| =', np.abs(h_modf[keep] - grid.without(out).ptdf()).max())
cut = [grid.names.index('L1-2'), grid.names.index('L1-5')]
print('cut set {L1-2, L1-5}: det =', modf.modf_2x2(h, grid.frm, grid.to, *cut)[1])

## Susceptance changes by Woodbury

A 20 % susceptance increase on transformer T5-6 as a rank-1 Woodbury correction, and the MODF as the special
case $\alpha = -1$.

In [ ]:
t = grid.names.index('T5-6')
h_w = modf.woodbury(h, grid.frm, grid.to, [t], [0.2])
print('max |H_woodbury - H_recomputed| =', np.abs(h_w - grid.scaled(t, 1.2).ptdf()).max())
print('flow on T5-6 before / after:', round(float(h[t] @ grid.p), 2), '/', round(float(h_w[t] @ grid.p), 2), 'MW')
print('alpha = -1 vs MODF:', np.abs(modf.woodbury(h, grid.frm, grid.to, out, [-1, -1]) - h_modf).max())

## Bus split (BSDF) and the cross-coupler flow

Split bus 2 (generator 40 MW, load 21.7 MW): L1-2 and L2-5 stay on bus A with the injections, L2-3 and
L2-4 move to bus B. The coupler flow $c = p_c P$ follows from KCL at bus A, and $F' = F + c\,\beta$.

In [ ]:
station = 1
bus_a = [grid.names.index('L1-2'), grid.names.index('L2-5')]
s, p_ext = compose.split(h, grid.frm, grid.to, grid.b, grid.p, station, bus_a)
ref = grid.split(station, bus_a)
print(f'cross-coupler flow c = {s.c:.2f} MW from bus A to bus B')
print('max |H_bsdf - H_recomputed| =', np.abs(s.h - ref.ptdf()).max())
print('max |beta - (H_:,i - H_:,b)| =', np.abs(s.beta - (ref.ptdf()[:, station] - ref.ptdf()[:, grid.n_bus])).max())
print('max |F + c beta - H P| =', np.abs(grid.flows() + s.c * s.beta - s.h @ p_ext).max())
for k in grid.branches_at(station):
    print(f'{grid.names[k]:6s} {grid.flows()[k]:8.2f} -> {s.f[k]:8.2f} MW   beta {s.beta[k]:+.4f}')

Reassigning the 40 MW generator to bus B changes the flows by $-\beta x$, and the bus-A entries of $\beta$,
oriented out of bus A, sum to one (Exercise 3).

In [ ]:
print(splits.reassign(grid, station, bus_a, 40.0))
print('sum of beta over bus A =', splits.beta_sum(grid, station, bus_a))

## Composition order

Split bus 4, disconnect L6-13, then screen N-1 with a busbar outage of bus 9, in ToOp's order and with the
disconnection first. The final flows agree; the stored $\beta$ and the cross-coupler flow do not.

In [ ]:
bus_a4 = [grid.names.index('L2-4'), grid.names.index('L4-5')]
disc, busbar = grid.names.index('L6-13'), 8
c1 = compose.compose(grid, 3, bus_a4, disc, busbar)
c2 = compose.compose(grid, 3, bus_a4, disc, busbar, split_first=False)
f_ref, n1_ref, f_bb_ref = compose.brute_force(grid, 3, bus_a4, disc, busbar)
keep = [k for k in range(grid.n_branch) if k != disc]
print('composed vs brute force: N-0', np.abs(c1.f[keep] - f_ref).max(), 'MW')
print('                         N-1', np.nanmax(np.abs(c1.n1[:, keep] - n1_ref)), 'MW')
print('order swapped: final flows differ by', np.abs(c1.f - c2.f).max(), 'MW')
print('               beta differs by', round(float(np.abs(c1.beta - c2.beta).max()), 4))
print(f'cross-coupler flow: {c1.c:.2f} MW (split first) vs {c2.c:.2f} MW (disconnection first)')